# Chatbot Académico EPIIS-UNSAAC

Demo interactiva del chatbot. Clona el repositorio, instala dependencias mínimas y prueba el clasificador de intents.

**Arquitectura:** basada en coincidencia de keywords + solapamiento Jaccard con trigger_phrases.  
Sin modelos externos — 100% Python estándar.

## 1. Configuración del entorno (solo en Colab)

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clona el repositorio y lo agrega al path
    !git clone https://github.com/Marcopolo988/epiis-knowledge.git
    sys.path.insert(0, '/content/epiis-knowledge')
    BASE_DIR = '/content/epiis-knowledge'
else:
    # Ejecución local: el notebook está en notebooks/, el repo es el padre
    from pathlib import Path
    BASE_DIR = str(Path.cwd().parent)
    sys.path.insert(0, BASE_DIR)

print(f'BASE_DIR = {BASE_DIR}')

## 2. Importar el chatbot

In [ ]:
from src.chatbot import ChatbotEPIIS

bot = ChatbotEPIIS(BASE_DIR)
print('Chatbot listo.')

## 3. Prueba básica

In [ ]:
preguntas_demo = [
    '¿Qué es la tutoría académica?',
    '¿Cuántos semestres dura la carrera?',
    '¿Qué requisitos necesito para iniciar mis prácticas?',
    '¿A qué países puedo ir de intercambio?',
    '¿Hay psicólogo gratis en la universidad?',
    'cuantas horas de practicas me piden',
    'oe explicame que es eso de tutoria xfa',
]

for pregunta in preguntas_demo:
    print(f'\n❓ {pregunta}')
    print(f'🤖 {bot.ask(pregunta)}')
    print('-' * 60)

## 4. Modo debug — ver intent y confianza

In [ ]:
import json

result = bot.ask_debug('puedo cambiarme de tutor?')
print(json.dumps(result, ensure_ascii=False, indent=2))

## 5. Evaluación sobre el corpus

In [ ]:
import json
from pathlib import Path

corpus_path = Path(BASE_DIR) / 'corpus' / 'corpus_consultas.json'
with open(corpus_path, encoding='utf-8') as f:
    corpus = json.load(f)

consultas = corpus['consultas']
total = len(consultas)
correctas = 0
errores = []

for c in consultas:
    result = bot.ask_debug(c['texto'])
    if result['intent'] == c['intent_esperado']:
        correctas += 1
    else:
        errores.append({
            'texto': c['texto'],
            'esperado': c['intent_esperado'],
            'obtenido': result['intent'],
            'confianza': result['confidence'],
        })

accuracy = correctas / total * 100
print(f'Accuracy: {correctas}/{total} = {accuracy:.1f}%')
print(f'Errores: {len(errores)}')

In [ ]:
# Detalle de errores
for e in errores:
    print(f"  Texto    : {e['texto']}")
    print(f"  Esperado : {e['esperado']}")
    print(f"  Obtenido : {e['obtenido']}  (conf={e['confianza']})")
    print()

## 6. Chat interactivo

In [ ]:
# Ejecuta esta celda y escribe tu pregunta en el campo que aparece.
# Escribe 'salir' para terminar.

while True:
    try:
        pregunta = input('Tú: ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if pregunta.lower() in ('salir', 'exit', 'quit'):
        print('Hasta luego.')
        break
    if pregunta:
        print(f'Bot: {bot.ask(pregunta)}\n')